In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
!pip install duckdb memory-profiler matplotlib pandas -q

import duckdb
import pandas as pd
import time
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from memory_profiler import memory_usage
from datetime import datetime
from pathlib import Path

DB_PATH  = '/content/drive/MyDrive/assignment/ecommerce.duckdb'
OCT_PATH = '/content/drive/MyDrive/assignment/archive/2019-Oct.csv'
NOV_PATH = '/content/drive/MyDrive/assignment/archive/2019-Nov.csv'

con = duckdb.connect(DB_PATH)
print("Connected ✓")

IOException: IO Error: Cannot open file "/content/drive/MyDrive/assignment/ecommerce.duckdb": No such file or directory

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/assignment/pipeline')
from extract import extract
from transform import transform
from load import load

def load_file(filepath, month_label):
    con_bench = duckdb.connect(DB_PATH)
    quality_log = {}
    t_start = time.perf_counter()

    for chunk, _, _ in extract(filepath, chunk_size=100_000):
        transformed = transform(chunk, quality_log)
        load(con_bench, transformed)

    elapsed = time.perf_counter() - t_start
    rows = con_bench.execute(
        f"SELECT COUNT(*) FROM fact_events WHERE event_month = '{month_label}'"
    ).fetchone()[0]
    con_bench.close()
    return elapsed, rows

print("Loading October...")
oct_time, oct_rows = load_file(OCT_PATH, '2019-10')

print("Loading November...")
nov_time, nov_rows = load_file(NOV_PATH, '2019-11')

total_time = oct_time + nov_time

print(f"\n{'File':<12} {'Rows':>12} {'Time (s)':>10} {'Rows/sec':>12}")
print("-" * 50)
print(f"{'2019-Oct':<12} {oct_rows:>12,} {oct_time:>10.1f} {oct_rows/oct_time:>12,.0f}")
print(f"{'2019-Nov':<12} {nov_rows:>12,} {nov_time:>10.1f} {nov_rows/nov_time:>12,.0f}")
print(f"{'TOTAL':<12} {oct_rows+nov_rows:>12,} {total_time:>10.1f} {(oct_rows+nov_rows)/total_time:>12,.0f}")

In [ ]:
# Load a 1M row sample into a temp table for batch testing
con.execute("""
    CREATE TABLE IF NOT EXISTS bench_source AS
    SELECT * FROM fact_events LIMIT 1_000_000
""")

BATCH_SIZES = [10_000, 50_000, 100_000, 500_000]
results = []

for batch_size in BATCH_SIZES:
    # fresh target table each run
    con.execute("DROP TABLE IF EXISTS bench_target")
    con.execute("CREATE TABLE bench_target AS SELECT * FROM fact_events LIMIT 0")

    total_rows = con.execute("SELECT COUNT(*) FROM bench_source").fetchone()[0]
    offset = 0
    t0 = time.perf_counter()

    while offset < total_rows:
        con.execute(f"""
            INSERT INTO bench_target
            SELECT * FROM bench_source
            LIMIT {batch_size} OFFSET {offset}
        """)
        offset += batch_size

    elapsed = time.perf_counter() - t0
    throughput = total_rows / elapsed
    results.append({
        'batch_size': batch_size,
        'time_sec':   round(elapsed, 2),
        'rows_per_sec': round(throughput, 0)
    })
    print(f"Batch {batch_size:>8,} → {throughput:>10,.0f} rows/sec  ({elapsed:.2f}s)")

con.execute("DROP TABLE IF EXISTS bench_target")
con.execute("DROP TABLE IF EXISTS bench_source")

batch_df = pd.DataFrame(results)
print("\n", batch_df.to_string(index=False))

In [ ]:
from extract import extract
from transform import transform
from load import load

def pipeline_chunk():
    con_m = duckdb.connect(DB_PATH)
    quality_log = {}
    # measure on first 500k rows only (representative sample)
    count = 0
    for chunk, _, _ in extract(OCT_PATH, chunk_size=100_000):
        transformed = transform(chunk, quality_log)
        load(con_m, transformed)
        count += len(chunk)
        if count >= 500_000:
            break
    con_m.close()

print("Measuring peak memory usage (this takes a minute)...")
mem = memory_usage(pipeline_chunk, interval=0.5, timeout=300)
peak_mb = max(mem)
baseline_mb = mem[0]

print(f"Baseline RAM : {baseline_mb:.1f} MB")
print(f"Peak RAM     : {peak_mb:.1f} MB")
print(f"Pipeline used: {peak_mb - baseline_mb:.1f} MB above baseline")

In [ ]:
import os

oct_size_mb = os.path.getsize(OCT_PATH) / 1024**2
nov_size_mb = os.path.getsize(NOV_PATH) / 1024**2
db_size_mb  = os.path.getsize(DB_PATH)  / 1024**2
raw_total   = oct_size_mb + nov_size_mb
ratio       = raw_total / db_size_mb

print(f"{'Source':<25} {'Size (MB)':>12}")
print("-" * 40)
print(f"{'2019-Oct.csv':<25} {oct_size_mb:>12.1f}")
print(f"{'2019-Nov.csv':<25} {nov_size_mb:>12.1f}")
print(f"{'Raw total':<25} {raw_total:>12.1f}")
print(f"{'ecommerce.duckdb':<25} {db_size_mb:>12.1f}")
print(f"{'Compression ratio':<25} {ratio:>11.2f}x")

In [ ]:
# The 3 queries we will test
QUERIES = {
    'Q1_funnel': """
        SELECT category_id, event_type, COUNT(DISTINCT user_id) AS users
        FROM fact_events
        GROUP BY category_id, event_type
    """,
    'Q3_top_brands': """
        SELECT p.brand, fe.event_month,
               SUM(fe.price) AS revenue
        FROM fact_events fe
        JOIN dim_product p ON fe.product_id = p.product_id
        WHERE fe.event_type = 'purchase'
        GROUP BY p.brand, fe.event_month
        ORDER BY revenue DESC
        LIMIT 10
    """,
    'Q5_hourly': """
        SELECT EXTRACT(hour FROM event_time) AS hour,
               COUNT(*) AS purchases
        FROM fact_events
        WHERE event_type = 'purchase'
        GROUP BY hour
        ORDER BY hour
    """
}

DROP_INDEXES = [
    "DROP INDEX IF EXISTS idx_events_event_type",
    "DROP INDEX IF EXISTS idx_events_user_id",
    "DROP INDEX IF EXISTS idx_events_product_id",
    "DROP INDEX IF EXISTS idx_events_event_time",
    "DROP INDEX IF EXISTS idx_events_month",
    "DROP INDEX IF EXISTS idx_events_category",
    "DROP INDEX IF EXISTS idx_events_cat_type",
    "DROP INDEX IF EXISTS idx_events_session",
]

CREATE_INDEXES = [
    "CREATE INDEX IF NOT EXISTS idx_events_event_type ON fact_events(event_type)",
    "CREATE INDEX IF NOT EXISTS idx_events_user_id    ON fact_events(user_id)",
    "CREATE INDEX IF NOT EXISTS idx_events_product_id ON fact_events(product_id)",
    "CREATE INDEX IF NOT EXISTS idx_events_event_time ON fact_events(event_time)",
    "CREATE INDEX IF NOT EXISTS idx_events_month      ON fact_events(event_month)",
    "CREATE INDEX IF NOT EXISTS idx_events_category   ON fact_events(category_id)",
    "CREATE INDEX IF NOT EXISTS idx_events_cat_type   ON fact_events(category_id, event_type)",
    "CREATE INDEX IF NOT EXISTS idx_events_session    ON fact_events(user_session, user_id)",
]

def time_query(con, sql, runs=3):
    """Average over multiple runs for stable timing."""
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        con.execute(sql).df()
        times.append(time.perf_counter() - t0)
    return round(sum(times) / len(times), 4)

index_results = []

# ── WITHOUT indexes ──
print("Dropping indexes...")
for stmt in DROP_INDEXES:
    con.execute(stmt)

print("Timing queries WITHOUT indexes...")
for name, sql in QUERIES.items():
    t = time_query(con, sql)
    index_results.append({'query': name, 'indexes': 'No', 'time_sec': t})
    print(f"  {name:<20} {t:.4f}s")

# ── WITH indexes ──
print("\nCreating indexes...")
for stmt in CREATE_INDEXES:
    con.execute(stmt)

print("Timing queries WITH indexes...")
for name, sql in QUERIES.items():
    t = time_query(con, sql)
    index_results.append({'query': name, 'indexes': 'Yes', 'time_sec': t})
    print(f"  {name:<20} {t:.4f}s")

idx_df = pd.DataFrame(index_results)
pivot  = idx_df.pivot(index='query', columns='indexes', values='time_sec')
pivot['speedup_x'] = (pivot['No'] / pivot['Yes']).round(2)
print("\nIndex impact summary:")
print(pivot.to_string())

In [ ]:
with open('/content/drive/MyDrive/assignment/reports/pipeline_run.json') as f:
    run_log = json.load(f)

q = run_log['quality_summary']
total     = run_log['total_extracted']
dropped   = q.get('rows_dropped', 0)
flagged   = q.get('negative_prices', 0) + q.get('large_prices', 0) + q.get('bad_timestamps', 0)
passed    = total - dropped
pass_rate = round(passed / total * 100, 2) if total > 0 else 0

print(f"{'Metric':<35} {'Count':>12}")
print("-" * 50)
print(f"{'Total rows extracted':<35} {total:>12,}")
print(f"{'Rows dropped (critical NULLs/dupes)':<35} {dropped:>12,}")
print(f"{'Rows flagged (price/date anomalies)':<35} {flagged:>12,}")
print(f"{'Duplicate events dropped':<35} {q.get('duplicates_dropped',0):>12,}")
print(f"{'Negative prices dropped':<35} {q.get('negative_prices',0):>12,}")
print(f"{'Large prices flagged (>10k)':<35} {q.get('large_prices',0):>12,}")
print(f"{'Bad timestamps flagged':<35} {q.get('bad_timestamps',0):>12,}")
print(f"{'Rows loaded successfully':<35} {passed:>12,}")
print(f"{'Overall pass rate':<35} {pass_rate:>11.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Performance report', fontsize=14, fontweight='500')

# ── Chart 1: Batch size vs throughput ──
ax1 = axes[0]
ax1.plot(
    [r['batch_size'] for r in results],
    [r['rows_per_sec'] for r in results],
    marker='o', linewidth=2, color='#7F77DD'
)
ax1.set_title('Batch size vs insert throughput', fontsize=12)
ax1.set_xlabel('Batch size (rows)')
ax1.set_ylabel('Rows / second')
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax1.grid(axis='y', alpha=0.3)

# ── Chart 2: Query time with vs without indexes ──
ax2 = axes[1]
query_names = pivot.index.tolist()
x = range(len(query_names))
width = 0.35
ax2.bar([i - width/2 for i in x], pivot['No'],  width, label='No index',   color='#F09595')
ax2.bar([i + width/2 for i in x], pivot['Yes'], width, label='With index', color='#5DCAA5')
ax2.set_title('Query time: index vs no index', fontsize=12)
ax2.set_ylabel('Time (seconds)')
ax2.set_xticks(list(x))
ax2.set_xticklabels(query_names, rotation=15, ha='right', fontsize=9)
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# ── Chart 3: Conversion funnel for top 5 categories ──
funnel_df = con.execute("""
    SELECT c.category_main,
           COUNT(DISTINCT CASE WHEN fe.event_type='view'     THEN fe.user_id END) AS views,
           COUNT(DISTINCT CASE WHEN fe.event_type='cart'     THEN fe.user_id END) AS carts,
           COUNT(DISTINCT CASE WHEN fe.event_type='purchase' THEN fe.user_id END) AS purchases
    FROM fact_events fe
    JOIN dim_category c ON fe.category_id = c.category_id
    WHERE c.category_main IS NOT NULL
    GROUP BY c.category_main
    ORDER BY views DESC
    LIMIT 5
""").df()

ax3 = axes[2]
bar_width = 0.25
cats = funnel_df['category_main'].tolist()
x3 = range(len(cats))
ax3.bar([i - bar_width for i in x3],   funnel_df['views'],     bar_width, label='View',     color='#AFA9EC')
ax3.bar([i             for i in x3],   funnel_df['carts'],     bar_width, label='Cart',     color='#7F77DD')
ax3.bar([i + bar_width for i in x3],   funnel_df['purchases'], bar_width, label='Purchase', color='#3C3489')
ax3.set_title('Conversion funnel — top 5 categories', fontsize=12)
ax3.set_ylabel('Distinct users')
ax3.set_xticks(list(x3))
ax3.set_xticklabels(cats, rotation=15, ha='right', fontsize=9)
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax3.legend()
ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/assignment/reports/performance_charts.png', dpi=150)
plt.show()
print("Chart saved ✓")